# 13 — Project Summary: Early Warning for Long Administrative Cases

**Author:** Ali Čikić · General Assembly – Data Science Bootcamp

**Question:** Can early information from an administrative process help identify cases likely to have a longer
**recorded activity span**?

This notebook contains **no code**. It is the starting point: it summarises notebooks 01–12, and every number below comes
from their saved outputs. Follow the links for the details.

## 1. Data and Target

- **Data:** BPI Challenge 2015, Municipality 1: building-permit cases from **one Dutch municipality** (52,217 events,
  1,199 cases). This is public research data, **not SPAR data**.
- **Historically completed cases:** cases with a recorded `case:endDate` (956 cases). A recorded end date does not
  guarantee that every step of the case is in the log.
- **Recorded activity span:** days from the **first** to the **last** recorded event of a case. It is **not** a verified
  administrative completion time, a legal deadline, a measure of lateness or a measure of employee performance. For 713 of
  the 956 completed cases the last recorded event comes after the recorded end date, and for 170 it comes before it.
- **Long case:** a recorded activity span of **more than 63 days** (472 of 956 completed cases). The 63 days are the median
  span of all completed cases in [01_EDA](01_EDA.ipynb). This is an **exploratory, data-derived threshold**: it is not an
  external deadline, and it was chosen with all completed cases, not learned from training cases only.

## 2. Reading Order

**Main story**

1. [13_project_summary](13_project_summary.ipynb) (this notebook)
2. [01_EDA](01_EDA.ipynb): the event log, data quality and the target
3. [06_early_warning_features_review](06_early_warning_features_review.ipynb): the first five events (exploratory step)
4. [07_14_day_early_warning_review](07_14_day_early_warning_review.ipynb): day-14 count features
5. [08_14_day_activity_features_review](08_14_day_activity_features_review.ipynb): day-14 counts plus activity indicators
6. [10_day30_early_warning_review](10_day30_early_warning_review.ipynb): day-30 features with cross-validation
7. [11_day30_temporal_validation_review](11_day30_temporal_validation_review.ipynb): train on older, test on newer cases
8. [12_day30_model_comparison_review](12_day30_model_comparison_review.ipynb): Logistic Regression vs Random Forest vs Gradient Boosting

**Supporting experiments**

- [02_linear_regression_review](02_linear_regression_review.ipynb), [03_decision_tree_review](03_decision_tree_review.ipynb),
  [04_random_forest_review](04_random_forest_review.ipynb), [05_gradient_boosting_review](05_gradient_boosting_review.ipynb):
  whole-case regression exercises
- [09_day14_regression_review](09_day14_regression_review.ipynb): predicting the span in days from day-14 features

## 3. How the Project Developed

| Step | Notebooks | What I tried | What I learned |
|---|---|---|---|
| 1 | [01](01_EDA.ipynb) | Load and explore the event log | Key columns complete; spans vary a lot (median 63 days, long tail) |
| 2 | [02](02_linear_regression_review.ipynb)–[05](05_gradient_boosting_review.ipynb) | Predict the span in days from **whole-case** features | Test MAE fell from 64.42 days (median baseline) to 51.19 (Gradient Boosting) on one split. These features exist only after a case ends, so these are **learning exercises, not early warnings** |
| 3 | [06](06_early_warning_features_review.ipynb) | Use only the first 5 events | Weak: accuracy 0.55 vs 0.51 for the baseline. 21 cases were already over 63 days at their fifth event, so a prediction for them is not a warning |
| 4 | [07](07_14_day_early_warning_review.ipynb) | Day-14 snapshot, 7 count features | No signal: accuracy 0.49 vs 0.52 for the baseline |
| 5 | [08](08_14_day_activity_features_review.ipynb) | Day-14 counts plus which activities were recorded | ROC-AUC 0.53 → 0.59: a small, exploratory gain |
| 6 | [09](09_day14_regression_review.ipynb) | Predict the span in days from day-14 features | No model beat the median baseline (MAE 70.75 vs 76.90–81.27) |
| 7 | [10](10_day30_early_warning_review.ipynb) | Day-30 snapshot with 5-fold cross-validation | Modest signal: Logistic Regression ROC-AUC 0.702 (cross-validation), 0.659 (test) |
| 8 | [11](11_day30_temporal_validation_review.ipynb) | Train on older cases, test on newer cases | Similar ranking (ROC-AUC 0.678), but lower recall (0.55) |
| 9 | [12](12_day30_model_comparison_review.ipynb) | Compare Random Forest and Gradient Boosting at day 30 | No consistent gain over Logistic Regression |

The tested day-14 approaches had **weak predictive signal**. This is a valid result: it led to the later day-30 snapshot.

## 4. How the Experiments Were Set Up

- **Eligibility.** At day N a case is included if its last recorded event is after first event + N days: day 14 gives
  917 cases, day 30 gives 838. The cohort is **retrospective**: *historically completed cases whose recorded activity span
  extends beyond day N*. It does not prove that a case was open at day N: 19 (day 14) and 39 (day 30) of these cases
  already have a recorded end date on or before the snapshot.
- **Three separate timing questions.** (1) **Feature timing:** the features use only events timestamped on or before the
  cutoff, and a leakage check confirms this in code. (2) **Cohort selection** uses each case's last event, which a live
  system would not know at the cutoff. (3) **Case status:** a real system would need to know which cases are open, and
  this log does not show that reliably. So the workflow is **not** free of future information just because the features are.
- **Activity indicators** are selected on training cases only, and again inside every cross-validation training fold.
  Scaling is fitted on training data only (inside a pipeline).
- **Random split:** created in notebook 07 (733 training / 184 test cases). Notebooks 08–10 and 12 reuse these exact
  groups; at day 30 they are filtered to 677 / 161, never redrawn.
- **Temporal split:** training on older cases and testing on newer ones (617 / 168, boundary 2013-10-25). Training cases
  must have their last event and end date before the boundary. 138 of the 168 temporal test cases were training cases in
  the random split, so this is a **chronological robustness check**, not an independent final test.
- **Reused test cases.** The same test cases were looked at in several notebooks, so all later results are
  **exploratory**, not an untouched final test.
- **Day 14 vs day 30** is not a perfectly controlled comparison: the eligible cases, the share of long cases (51.5% at day
  14; 55.7% and 59.0% in the day-30 training and test groups) and the feature window all change.
- **Redundant features.** Two of the seven count features are exact combinations of others (repeated activities = events −
  unique activities; events in the second interval = all events − events in the first interval). They are kept for
  continuity and their separate coefficients are not interpreted.

## 5. Main Day-30 Comparison

**ROC-AUC** measures how well a model ranks long cases above not-long cases: 0.50 is no better than chance.

| ROC-AUC | Cross-validation mean | Random test (161 cases) | Temporal test (168 cases) |
|---|---|---|---|
| Majority baseline | 0.500 | 0.500 | 0.500 |
| Logistic Regression | 0.702 | 0.659 | 0.678 |
| Random Forest | 0.701 | 0.745 | 0.674 |
| Gradient Boosting | 0.684 | 0.723 | 0.624 |

Notebook 12 shows this as a chart (with ±1 standard deviation across the five folds, which is not a confidence interval)
and shows Logistic Regression's temporal-test confusion matrix.

**Why Logistic Regression stays the main candidate:** Random Forest was strongest on the random test (0.745), but it was
practically equal to Logistic Regression in cross-validation (0.701 vs 0.702) and on the temporal test (0.674 vs 0.678).
One favourable test result does not show that a model is better. Random Forest's warnings also changed more between the
tests: it flagged 118 of 161 random-test cases (recall 0.863) but only 51 of 168 temporal-test cases (recall 0.43). When
the results are similar, the simpler and easier-to-explain model is preferred.

## 6. What the Baseline Means

The **majority baseline** predicts "long" for every case, because long cases are the more common class at day 30. It
therefore finds **every** long case (recall 1.00) and gets a high F1, but it cannot tell cases apart (ROC-AUC 0.50).
That is why recall and F1 alone can make a useless model look good. A warning model must **rank** cases better and give
warnings that are more often correct:

| Test | Model | Cases flagged | Warnings correct (precision) | Long cases found (recall) |
|---|---|---|---|---|
| Random test (161 cases) | Majority baseline | 161 | 59.0% | 100% |
| Random test (161 cases) | Logistic Regression | 98 | 69.4% | 71.6% |
| Temporal test (168 cases) | Majority baseline | 168 | 59.5% | 100% |
| Temporal test (168 cases) | Logistic Regression | 71 | 77.5% | 55% |

On the temporal test, Logistic Regression's warnings are more often correct than random flagging, but it misses 45 of the
100 long cases.

## 7. Conclusion

- The tested **day-14** approaches showed **weak** predictive signal.
- At **day 30**, Logistic Regression shows a **modest ranking signal** (ROC-AUC about 0.66–0.70 across cross-validation,
  the random test and the temporal test).
- Its **warning performance is unstable**: recall was 0.716 on the random test but 0.55 on the newer cases.
- The evidence is **not sufficient for operational use**: one municipality, a retrospective cohort, reused test cases and a
  data-derived threshold.

These conclusions are about the tested features, models and sample. A low ROC-AUC for one feature set does not prove that
all early information is uninformative, and feature importances or coefficients do not show causes.

## 8. Limitations

- The target is the **recorded activity span**, not a verified completion time; the recorded end date often differs from
  the last recorded event.
- Only **historically completed** cases are included, and eligibility at day 14/30 is defined retrospectively.
- The 63-day threshold was chosen during exploratory analysis of **all** completed cases.
- Timestamps are assumed to show when information was available; the leakage checks verify the feature code, not this
  assumption.
- Test cases were **reused** across notebooks, and the temporal test overlaps with earlier training cases.
- **One municipality**; a similar ROC-AUC on newer cases does not prove that the model generalises or would be useful in practice.
- Forecasting the number of new cases over time was **not** part of this project.

## 9. Connection to SPAR

This project shows a **method that might later be evaluated on SPAR's own historical data**: build features from what is
known at a fixed early moment, compare them with a simple baseline, and check the result on newer cases. It does **not**
establish any performance for SPAR or Montenegro, and its results cannot be transferred directly.